In [ ]:
import pandas as pd
import hashlib
import requests
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

**TEST DATASET**

In [ ]:
df = pd.read_csv("/content/Marketingcampaigns.csv")
X = df.drop("Purchased", axis=1)
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

**TEST MODEL**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#  Feature Engineering (boost performance)
X['Engagement Score'] = X['Email Opened'] + X['Email Clicked'] + X['Product page visit']
X['Interaction'] = X['Age'] * X['Discount offered']

# Train-test split (stratified for better testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Drop unnecessary column
X_train = X_train.drop('Customer id', axis=1)
X_test = X_test.drop('Customer id', axis=1)

# Feature types
numeric_features = [
    'Age', 'Email Opened', 'Email Clicked',
    'Product page visit', 'Discount offered',
    'Engagement Score', 'Interaction'
]

categorical_features = ['Gender', 'Location']

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
])

# Pipeline with Logistic Regression
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=5000))
])

#  Strong hyperparameter tuning for Logistic Regression
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'liblinear'],
    'model__class_weight': [None, 'balanced']
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predictions
preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, preds)

print("Best Params:", grid.best_params_)
print("Accuracy:", accuracy)
print("Sample Predictions:", preds[:5])
print("Sample Probabilities:", probs[:5])

Best Params: {'model__C': 0.001, 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Accuracy: 0.75
Sample Predictions: [0 0 1 0]
Sample Probabilities: [0.49713809 0.49759317 0.50127635 0.49804319]


 SAVE MODEL, GENERATE HASH FUNCTION and **COMMIT FUNCTION**

In [ ]:
import hashlib
import requests
import json

BASE_URL = "https://transocular-shrewishly-joie.ngrok-free.dev"


# 🔹 Hash function
def hash_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# 🔹 Metadata hash
def hash_metadata(metadata_dict):
    metadata_str = json.dumps(metadata_dict, sort_keys=True)
    return hashlib.sha256(metadata_str.encode()).hexdigest()


# 🔹 Get previous record
def get_previous_record():
    try:
        res = requests.get(f"{BASE_URL}/latest", timeout=5)
        if res.status_code == 200:
            data = res.json()
            if "dataset_hash" in data:
                return data
    except Exception as e:
        print("⚠ Could not fetch previous record:", e)

    return None


# FINAL COMMIT FUNCTION
def commit_model(dataset_path, model_path, accuracy, model_type="Unknown", params=None, features=None):

    print("\n Starting commit...")

    # 🔹 Step 1: Upload files
    try:
        with open(model_path, "rb") as m, open(dataset_path, "rb") as d:

            files = {
                "model": m,
                "dataset": d
            }

            upload_res = requests.post(f"{BASE_URL}/upload", files=files, timeout=10)

        if upload_res.status_code != 200:
            print(" Upload failed:", upload_res.text)
            return

        upload_data = upload_res.json()

        model_hash = upload_data["model_hash"]
        dataset_hash = upload_data["dataset_hash"]

        print("Files uploaded successfully")
        print("Model Hash:", model_hash)
        print("Dataset Hash:", dataset_hash)

    except Exception as e:
        print(" Upload error:", e)
        return


    #  Step 2: Create metadata (NEW)
    model_metadata = {
        "model_type": model_type,
        "params": params if params else {},
        "features": features if features else []
    }

    metadata_hash = hash_metadata(model_metadata)

    print("Metadata Hash:", metadata_hash)


    # 🔹 Step 3: Get previous record
    prev = get_previous_record()


    # 🔹 Step 4: Change Detection
    if prev:
        dataset_changed = dataset_hash != prev.get("dataset_hash")
        model_changed = model_hash != prev.get("model_hash")

        # NEW: Metadata change detection
        metadata_changed = metadata_hash != prev.get("metadata_hash")

        prev_acc = float(prev.get("accuracy", 0))
        acc_diff = float(accuracy) - prev_acc

        print("\n CHANGE SUMMARY")
        print("Dataset Changed:", dataset_changed)
        print("Model Changed:", model_changed)
        print("Metadata Changed:", metadata_changed)
        print("Accuracy Change:", round(acc_diff, 4))

        #  Improved logic
        if not dataset_changed and not model_changed and not metadata_changed:
            print("⚠ No changes detected — skipping blockchain commit")
            return

        if dataset_changed and model_changed:
            print("Status: Normal Update ")
        elif dataset_changed:
            print("Status: Data Drift ⚠")
        elif model_changed:
            print("Status: Model Updated ")
        elif metadata_changed:
            print("Status: Model Logic Changed ")

        if acc_diff < -0.02:
            print("Accuracy dropped!")

    else:
        print(" First Version (Genesis)")


    # 🔹 Step 5: Store on blockchain
    data = {
        "dataset_hash": dataset_hash,
        "model_hash": model_hash,
        "metadata_hash": metadata_hash,
        "accuracy": str(accuracy),
        "model_type": model_type
    }

    try:
        res = requests.post(f"{BASE_URL}/store", json=data, timeout=10)

        print("\nServer response status:", res.status_code)
        print(" Raw response:", res.text)

        try:
            print("Model committed:", res.json())
        except:
            print("⚠ Response is not JSON")

    except Exception as e:
        print("Connection error:", e)

In [ ]:
import joblib

#  Step 1: Save the CORRECT model
MODEL_PATH = "model.pkl"
DATASET_PATH = "/content/Marketingcampaigns.csv"   # adjust if needed

#  IMPORTANT: save best_model, NOT model
joblib.dump(best_model, MODEL_PATH)

print(" Model saved locally")


# 🔹 Step 2: Commit (handles upload + store internally)
commit_model(
    dataset_path=DATASET_PATH,
    model_path=MODEL_PATH,
    accuracy=accuracy,
    params=grid.best_params_,           # from GridSearch
    features=list(X.columns)            # after feature engineering
)


 Model saved locally

 Starting commit...
Files uploaded successfully
Model Hash: 1acb543b62de6ef4e959ba3e904127b8d4133f4c599e3685222376437f2ca862
Dataset Hash: 2a36f21afdd69f8e12c0bc384e5c82336a3b2b43485cde91c9ae20d88e8d368f
Metadata Hash: 9ba9c810074c342de4524859e1ea1ef5b654754d196e2f43c8217483982aeeb9

 CHANGE SUMMARY
Dataset Changed: False
Model Changed: True
Metadata Changed: True
Accuracy Change: 0.0
Status: Model Updated 

Server response status: 200
 Raw response: {"mined_block":null,"pending_transactions":1,"status":"stored","tx_hash":"67a4fdfba0b4d0b43555086072617783f21906cf79cecb84144b28f6e5afe9a6"}

Model committed: {'mined_block': None, 'pending_transactions': 1, 'status': 'stored', 'tx_hash': '67a4fdfba0b4d0b43555086072617783f21906cf79cecb84144b28f6e5afe9a6'}
